# NCAA 2026

**Resources**
- [Predicting bracket](https://towardsdatascience.com/data-driven-march-madness-predictions/)


In [296]:
import re
import pandas as pd 
from pathlib import Path 

INPUT_PATH = Path('/kaggle/input/competitions/march-machine-learning-mania-2026')
  
class NCADataset:
    def __init__(self, name: str):
        self.name = name
        # self.cities = pd.read_csv(INPUT_PATH / 'Cities.csv')
        # self.conferences = pd.read_csv(INPUT_PATH / 'Conferences.csv')
        
    def __setattr__(self, name, value):
        if not hasattr(self, name):
            print(f"NCAA Attribute {name} Added [{list(value.columns) if isinstance(value, pd.DataFrame) else len(value)}]")
        else:
            print(f"Attempted to add attribute {name} but already exists")
        super().__setattr__(name, value)

    @property
    def dataframes(self) -> dict:
        return {k: v for k, v in self.__dict__.items() if isinstance(v, pd.DataFrame)}

    def find_column(self, col: str) -> list:
        return [name for name, df in self.dataframes.items() if col in df.columns]

    @property
    def list_dataset(self):
        return [k for k, v in self.__dict__.items() if isinstance(v, pd.DataFrame)]

    def find_dataset(self, term: str):
        return [k for k, v in self.__dict__.items() if isinstance(v, pd.DataFrame) and term in k]

def find_relation(ds: NCADataset, target_df: str):
    """ Find mergable tables based on the shared table column fields. """
    
    for idx, (label, df) in enumerate(ds.dataframes.items(), start=1):
        if label != target_df:
            col = list(getattr(ds, target_df).columns.intersection(df.columns))
            if col is not None and len(col) > 1:
                print(f"{idx}. {label}\t {col}")
                
    print("Finished searching for table relations")
    
    
def load_dataset(for_male: bool = True):
    g_parse = "M*.csv" if for_male is True else "W*.csv"
    
    data = NCADataset(name="male" if for_male else "female")
    for f in INPUT_PATH.rglob(g_parse):
        file_name = f.stem
        name_term = re.findall(r'[A-Z][a-z]+', file_name)
        if len(name_term) > 0:
            name = "_".join(name_term).lower()
            setattr(data, name, pd.read_csv(f))
        else:
            raise ValueError 
        
    return data

male = load_dataset()

NCAA Attribute name Added [4]
NCAA Attribute tourney_seed_round_slots Added [['Seed', 'GameRound', 'GameSlot', 'EarlyDayNum', 'LateDayNum']]
NCAA Attribute regular_season_detailed_results Added [['Season', 'DayNum', 'WTeamID', 'WScore', 'LTeamID', 'LScore', 'WLoc', 'NumOT', 'WFGM', 'WFGA', 'WFGM3', 'WFGA3', 'WFTM', 'WFTA', 'WOR', 'WDR', 'WAst', 'WTO', 'WStl', 'WBlk', 'WPF', 'LFGM', 'LFGA', 'LFGM3', 'LFGA3', 'LFTM', 'LFTA', 'LOR', 'LDR', 'LAst', 'LTO', 'LStl', 'LBlk', 'LPF']]
NCAA Attribute tourney_compact_results Added [['Season', 'DayNum', 'WTeamID', 'WScore', 'LTeamID', 'LScore', 'WLoc', 'NumOT']]
NCAA Attribute game_cities Added [['Season', 'DayNum', 'WTeamID', 'LTeamID', 'CRType', 'CityID']]
NCAA Attribute seasons Added [['Season', 'DayZero', 'RegionW', 'RegionX', 'RegionY', 'RegionZ']]
NCAA Attribute secondary_tourney_teams Added [['Season', 'SecondaryTourney', 'TeamID']]
NCAA Attribute team_spellings Added [['TeamNameSpelling', 'TeamID']]
NCAA Attribute regular_season_compact_res

# Processing Draft 2

**Config Dataset**

- teams
- seasons
- game_cities 

**Game By Game**

- Day 134-135 → First Four (play-in games)
- Day 136-137 → Round of 64
- Day 138-139 → Round of 32
- Day 143-144 → Sweet 16
- Day 145-146 → Elite 8
- Day 152     → Final Four
- Day 154     → Championship

```bash
df['col'].fillna(0)          # fill with a value
df['col'].fillna(method='ffill')  # forward fill
df['col'].fillna(df['col'].mean())  # fill with mean
df['col'].dropna()           # drop nulls
df['col'].isna().sum()       # count nulls

In [513]:
# Loading & Processing Dataset Individually 

_GAME_ID = ['Season', 'DayNum', 'WTeamID', 'LTeamID', 'CityID', 'WSeed', 'LSeed']
# _GAME_PRE_FEATS = _GAME_ID + _GAME_FEATS
# _GAME_POST_FEATS = _GAME_ID + []
_PRED_FEATS = ['ID', '_pred_label']

def _restack_winners_losers(df: pd.DataFrame):
    assert 'LSeed' in df.columns and 'WSeed' in df.columns
    
    winners = df.assign(
        ID=df['ID'],
        Season=df['Season'], 
        Seed=df['WSeed'],
        DayNum = df['DayNum'],
        TeamID   = df['WTeamID'],
        OppID    = df['LTeamID'],
        Won      = 1,
        PtsFor   = df['WScore'],
        PtsAgn   = df['LScore'],
        FGM=df['WFGM'], FGA=df['WFGA'],
        FGM3=df['WFGM3'], FGA3=df['WFGA3'],
        FTM=df['WFTM'], FTA=df['WFTA'],
        OR=df['WOR'], DR=df['WDR'],
        Ast=df['WAst'], TO=df['WTO'],
        Stl=df['WStl'], Blk=df['WBlk'], PF=df['WPF'],
        OppFGM=df['LFGM'], OppFGA=df['LFGA'],
        OppOR=df['LOR'], OppDR=df['LDR'], OppTO=df['LTO'],
    )

    losers = df.assign(
        ID=df['ID'],
        Season=df['Season'],
        DayNum=df['DayNum'],
        Seed=df['LSeed'],
        TeamID   = df['LTeamID'],
        OppID    = df['WTeamID'],
        Won      = 0,
        PtsFor   = df['LScore'],
        PtsAgn   = df['WScore'],
        FGM=df['LFGM'], FGA=df['LFGA'],
        FGM3=df['LFGM3'], FGA3=df['LFGA3'],
        FTM=df['LFTM'], FTA=df['LFTA'],
        OR=df['LOR'], DR=df['LDR'],
        Ast=df['LAst'], TO=df['LTO'],
        Stl=df['LStl'], Blk=df['LBlk'], PF=df['LPF'],
        OppFGM=df['WFGM'], OppFGA=df['WFGA'],
        OppOR=df['WOR'], OppDR=df['WDR'], OppTO=df['WTO'],
    )

    cols = ['ID', 'Season', 'TeamID', 'DayNum','Won',
            'PtsFor', 'PtsAgn', 'Seed',
            'FGM', 'FGA', 'FGM3', 'FGA3', 'FTM', 'FTA',
            'OR', 'DR', 'Ast', 'TO', 'Stl', 'Blk', 'PF',
            'OppFGM', 'OppFGA', 'OppOR', 'OppDR', 'OppTO']

    return pd.concat([winners[cols], losers[cols]], ignore_index=True)

def _build_features_on_box_scores(games: pd.DataFrame):
    
    games['Poss']    = games['FGA'] - games['OR'] + games['TO'] + 0.44 * games['FTA']
    games['OppPoss'] = games['OppFGA'] - games['OppOR'] + games['OppTO'] + 0.44 * games['FTA']
    poss             = games['Poss'].clip(1)
    games['eFGPct']      = (games['FGM'] + 0.5 * games['FGM3']) / games['FGA'].clip(1)
    games['TSPct']       = games['PtsFor'] / (2 * (games['FGA'] + 0.44 * games['FTA'])).clip(1)
    games['FG3Pct']      = games['FGM3'] / games['FGA3'].clip(1)
    games['FTPct']       = games['FTM'] / games['FTA'].clip(1)
    games['FTRate']      = games['FTA'] / games['FGA'].clip(1)
    games['3PARate']     = games['FGA3'] / games['FGA'].clip(1)
    games['ORBPct']      = games['OR'] / (games['OR'] + games['OppDR']).clip(1)
    games['DRBPct']      = games['DR'] / (games['DR'] + games['OppOR']).clip(1)
    games['TOVRate']     = games['TO'] / poss
    games['AstTORatio']  = games['Ast'] / games['TO'].clip(1)
    games['AstRate']     = games['Ast'] / games['FGM'].clip(1)
    games['StlRate']     = games['Stl'] / games['OppPoss'].clip(1)
    games['BlkRate']     = games['Blk'] / games['OppFGA'].clip(1)
    games['OffEff']      = (games['PtsFor'] / poss) * 100
    games['DefEff']      = (games['PtsAgn'] / poss) * 100
    games['NetEff']      = games['OffEff'] - games['DefEff']
    games['ScoreMargin'] = games['PtsFor'] - games['PtsAgn']
    games['FourFactors'] = (
        0.40 * games['eFGPct']
      - 0.25 * games['TOVRate']
      + 0.20 * games['ORBPct']
      + 0.15 * games['FTRate']
    )
    return games 

def _build_features_on_box_scores_v2(df: pd.DataFrame) -> pd.DataFrame:
    """
    Creates features based on game box scores and play style for both Winner (W) and Loser (L). 
    Returns 
        Feature vector describing the play style of teams. Note that most of the metrics are relative to the Winner as we are more interested in the winning team's playstyle.

    """
    
    f = df.copy()
    original_cols = set(f.columns)
    # By Mi
    f['ScoreDiff'] = f.WScore - f.LScore
    f['ScoreRatio'] = (f.WScore - f.LScore) / f.WScore
    f['ScoreRatioProbInv'] = 1 - f['ScoreRatio']
    # ── 1. SHOOTING EFFICIENCY ───────────────────────────────────
    # Effective Field Goal % — accounts for 3pt being worth more
    # eFG% = (FGM + 0.5 * FGM3) / FGA
    f['W_eFGPct']       = (f['WFGM'] + 0.5 * f['WFGM3']) / f['WFGA'].clip(1)
    f['L_eFGPct']       = (f['LFGM'] + 0.5 * f['LFGM3']) / f['LFGA'].clip(1)
    f['eFGPCt'] = f['W_eFGPct'] - f['L_eFGPct']
    # True Shooting % — includes free throws
    # TS% = PTS / (2 * (FGA + 0.44 * FTA))
    f['W_TSPct']        = f['WScore'] / (2 * (f['WFGA'] + 0.44 * f['WFTA'])).clip(1)
    f['L_TSPct']        = f['LScore'] / (2 * (f['LFGA'] + 0.44 * f['LFTA'])).clip(1)
    f['TSPct'] = f['W_TSPct'] - f['L_TSPct']  
    # 3pt attempt rate — how much they rely on 3s
    f['W_3PARate']      = f['WFGA3'] / f['WFGA']
    f['L_3PARate']      = f['LFGA3'] / f['LFGA']
    f['3PARate'] = (f['W_3PARate'] - f['L_3PARate']) / f['W_3PARate']
    # 3pt accuracy
    f['W_3PPct']        = f['WFGM3'] / f['WFGA3'].clip(1)
    f['L_3PPct']        = f['LFGM3'] / f['LFGA3'].clip(1)
    f['3PPct'] = (f['W_3PPct'] + f['L_3PPct']) / f['W_3PPct']
    f['3PPctRatio'] = f['W_3PPct']/f['L_3PPct']
    # Free throw accuracy
    f['W_FTPct']        = f['WFTM'] / f['WFTA'].clip(1)
    f['L_FTPct']        = f['LFTM'] / f['LFTA'].clip(1)
    f['FTPct'] = (f['W_FTPct']  + f['L_FTPct'] ) / f['W_FTPct'] 
    f['FTPctRatio'] = f['W_FTPct'] / f['L_FTPct'] 
    # Free throw rate — how often they get to the line
    f['W_FTRate']       = f['WFTM'] / f['WFGA'].clip(1)
    f['L_FTRate']       = f['LFTM'] / f['LFGA'].clip(1)
    f['FTRate'] = f['W_FTRate'] / f['L_FTRate']
    # ── 2. REBOUNDING ────────────────────────────────────────────
    # Total rebounds
    f['W_TotalReb']     = f['WOR'] + f['WDR']
    f['L_TotalReb']     = f['LOR'] + f['LDR']
    f['TotalReb'] = f['W_TotalReb'] / f['L_TotalReb'] 
    # Offensive rebound rate — second chance opportunities
    # ORB% = ORB / (ORB + opp DRB)
    f['W_ORBRate']      = f['WOR'] / (f['WOR'] + f['LDR']).clip(1)
    f['L_ORBRate']      = f['LOR'] / (f['LOR'] + f['WDR']).clip(1)
    f['ORBRate'] = f['W_ORBRate'] / f['L_ORBRate']
    # Defensive rebound rate
    f['W_DRBRate']      = f['WDR'] / (f['WDR'] + f['LOR']).clip(1)
    f['L_DRBRate']      = f['LDR'] / (f['LDR'] + f['WOR']).clip(1)
    f['DRBRate'] = f['W_DRBRate'] / f['L_DRBRate']
    # Rebound margin
    f['RebMargin']      = f['W_TotalReb'] - f['L_TotalReb']
    f['RebMarginRatio'] =  (f['W_TotalReb'] - f['L_TotalReb']) /  f['W_TotalReb']
    f['RebMarginRatioNorm'] = (f['W_TotalReb'] - f['L_TotalReb']) /  (f['W_TotalReb'] + f['L_TotalReb'])
    
    # ── 3. BALL SECURITY / TURNOVERS ─────────────────────────────
    # Possessions estimate (Pace proxy)
    # Poss ≈ FGA - OR + TO + 0.44 * FTA
    f['W_Poss']         = f['WFGA'] - f['WOR'] + f['WTO'] + 0.44 * f['WFTA']
    f['L_Poss']         = f['LFGA'] - f['LOR'] + f['LTO'] + 0.44 * f['LFTA']
    f['Poss'] = f['W_Poss'] / f['L_Poss']
    # Turnover rate — turnovers per possession
    f['W_TOVRate']      = f['WTO'] / f['W_Poss'].clip(1)
    f['L_TOVRate']      = f['LTO'] / f['L_Poss'].clip(1)
    f['TOVRate'] = f['W_TOVRate']/f['L_TOVRate']
    # Assist to turnover ratio — ball movement quality
    f['W_AstTORatio']   = f['WAst'] / f['WTO'].clip(1)
    f['L_AstTORatio']   = f['LAst'] / f['LTO'].clip(1)
    f['AstTORatio'] = f['W_AstTORatio'] / f['L_AstTORatio']
    # Assist rate — % of FGM assisted
    f['W_AstRate']      = f['WAst'] / f['WFGM']
    f['L_AstRate']      = f['LAst'] / f['LFGM']
    f['AstRate'] = f['W_AstRate']/f['L_AstRate'].clip(1)
    
    # ── 4. DEFENSIVE PRESSURE ────────────────────────────────────
    # Steal rate — steals per opponent possession
    f['W_StlRate']      = f['WStl'] / f['L_Poss'].clip(1)
    f['L_StlRate']      = f['LStl'] / f['W_Poss'].clip(1)
    f['StlRate'] = f['W_StlRate']/f['L_StlRate']
    # Block rate — blocks per opponent FGA
    f['W_BlkRate']      = f['WBlk'] / f['LFGA'].clip(1)
    f['L_BlkRate']      = f['LBlk'] / f['WFGA'].clip(1)
    f['BlkRate'] = f['W_BlkRate']/f['L_BlkRate'].clip(1)
    # Foul rate — personal fouls per possession
    f['W_FoulRate']     = f['WPF'] / f['W_Poss'].clip(1)
    f['L_FoulRate']     = f['LPF'] / f['L_Poss'].clip(1)
    f['FoulRate'] = f['W_FoulRate']/f['L_FoulRate']
    # ── 5. EFFICIENCY (KenPom-style) ─────────────────────────────
    # Offensive efficiency — points per 100 possessions
    f['W_OffEff']       = (f['WScore'] / f['W_Poss'].clip(1))
    f['L_OffEff']       = (f['LScore'] / f['L_Poss'].clip(1))
    f['OffEff'] = f['W_OffEff'] / f['L_OffEff']
    # Defensive efficiency — points allowed per 100 possessions
    f['W_DefEff']       = (f['LScore'] / f['W_Poss'].clip(1)) 
    f['L_DefEff']       = (f['WScore'] / f['L_Poss'].clip(1))
    f['DefEff'] = f['W_DefEff']/f['L_DefEff']
    # Net efficiency — the single most predictive KenPom metric
    f['W_NetEff']       = f['W_OffEff'] - f['W_DefEff']
    f['L_NetEff']       = f['L_OffEff'] - f['L_DefEff']
    f['NetEff'] = f['W_NetEff']/f['L_NetEff']
    # Efficiency ratio
    f['W_EffRatio']     = f['W_OffEff'] / f['W_DefEff'].clip(1)
    f['L_EffRatio']     = f['L_OffEff'] / f['L_DefEff'].clip(1)
    f['EffRatio'] = f['W_EffRatio'] / f['L_EffRatio']
    # ── 6. PACE ──────────────────────────────────────────────────
    # Game pace — average possessions of both teams
    f['Pace']           = (f['W_Poss'] + f['L_Poss']) / 2
    # Overtime flag
    f['HadOT']          = (f['NumOT'] > 0).astype(int)
    # ── 7. SCORE / MARGIN ────────────────────────────────────────
    f['ScoreMargin']    = f['WScore'] - f['LScore']
    f['TotalPoints']    = f['WScore'] + f['LScore']

    # ── 8. FOUR FACTORS (Dean Oliver) ────────────────────────────
    # The four factors composite score (weighted per Dean Oliver) that determine wins:
    # 1. Shooting (eFG%)          — already computed above
    # 2. Turnovers (TOV rate)     — already computed above
    # 3. Rebounding (ORB rate)    — already computed above
    # 4. Free throws (FT rate)    — already computed above
    f['W_FourFactors']  = (
        0.40 * f['W_eFGPct']   +
        0.25 * (1 - f['W_TOVRate']) +   # lower TO = better
        0.20 * f['W_ORBRate']  +
        0.15 * f['W_FTRate']
    )
    f['L_FourFactors']  = (
        0.40 * f['L_eFGPct']   +
        0.25 * (1 - f['L_TOVRate']) +
        0.20 * f['L_ORBRate']  +
        0.15 * f['L_FTRate']
    )
    f['FourFactors'] = f['W_FourFactors'] / f['L_FourFactors']
    new_features = [c for c in f.columns if c not in original_cols]
    display(f'Added new game based features: \n\n{new_features}')
    
    return f[_GAME_ID + new_features + ['WSeed', 'LSeed']]

def create_team_score_stats_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Aggregates regular season box score stats per team per season
    to predict tournament seed.
    Input : regular season detailed results (one row per game)
    Output: one row per (Season, TeamID, ID) with aggregated features
    """
    # Stack W and L into team-centric rows
    games = _restack_winners_losers(df) 
    games = _build_features_on_box_scores(games)
    # Aggregate per team per season
    agg_cols = [
        'Won', 'PtsFor', 'PtsAgn', 'ScoreMargin',
        'eFGPct', 'TSPct', 'FG3Pct', 'FTPct', 'FTRate', '3PARate',
        'ORBPct', 'DRBPct', 'TOVRate', 'AstTORatio', 'AstRate',
        'StlRate', 'BlkRate', 'OffEff', 'DefEff', 'NetEff',
        'FourFactors', 'Poss',
    ]
    team_stats = (
        games.groupby(['Season', 'TeamID', 'ID'])[agg_cols]
        .agg(['mean', 'std', 'median'])
    )
    team_stats.columns = ['_'.join(c) for c in team_stats.columns]
    team_stats = team_stats.reset_index()
    win_agg = games.groupby(['Season', 'TeamID'])['Won'].agg(['mean', 'count']).reset_index()
    win_agg.columns = ['Season', 'TeamID', 'WinRate', 'NumGames']
    team_stats = team_stats.merge(win_agg, on=['Season', 'TeamID'], how='left')
    # Consistency score: low std on NetEff = more reliable team
    team_stats['NetEff_consistency'] = 1 / (team_stats['NetEff_std'] + 1e-5)

    return team_stats

def load_regular_games(ds: NCADataset):
    assert max(ds.regular_season_detailed_results.DayNum.unique()) <= 132
    
    regular = ds.regular_season_detailed_results.copy()
    regular['CRType'] = 'Regular'
    regular['ID'] = regular["Season"].astype(str) + "_" + regular["WTeamID"].astype(str) + "_" + regular["LTeamID"].astype(str)

    result = create_team_score_stats_features(regular)
    rename_map = {col: f'reg_{col}'.lower() for col in result.columns if col not in ('Season', 'TeamID', 'ID', 'CRType', 'WinRate', 'Won', 'DayNum')}
    result = result.rename(columns=rename_map)
    
    return result


In [514]:
# Main Entry point to prepare basic X,y Model Features

SYS_BOOKIES = ['MOR', 'POM', 'DOK', 'SAG', 'MAS', 'WLK', 'WIL', 'DOL']

def prepare_y_dataset(ds: NCADataset):
    """ Prepares and processes the Results (Y) data for model consumption. Initial preprocessing stage before segmenting into round level or season level or game (reg/ncaa/post-game) level. 
    
    Results dataset:
    [
        'regular_season_detailed_results',
        'tourney_compact_results',
        'regular_season_compact_results',
        'secondary_tourney_compact_results',
        'tourney_detailed_results'
    ]
    """
    
    assert min(ds.tourney_detailed_results.DayNum.unique()) >= 134 and max(ds.tourney_detailed_results.DayNum.unique()) <= 154
    tourney = ds.tourney_detailed_results.copy()
    tourney['CRType'] = 'NCAA'
    tourney_seed_slots = (
        male.tourney_seeds.merge(
            male.tourney_seed_round_slots.rename(columns={'GameSlot': 'Slot'}), 
            on=['Seed'], 
            how='left'
        )
        .merge(
            male.tourney_slots,
            left_on=['Season', 'Slot'],
            right_on=['Season', 'Slot'], 
            how='left'
        )
    )
    y = (
        tourney
        .merge(
            male.tourney_seeds.rename(columns={'TeamID': 'WTeamID'}),
            on=['Season', 'WTeamID'],
            how='left'
        )
        .rename(columns={'Seed': 'WSeed'})
        .merge(
            tourney_seed_slots.rename(columns={'TeamID': 'LTeamID'}),
            on=['Season', 'LTeamID'],
            how='left'
        )
        .rename(columns={'Seed': 'LSeed'})
        .assign(StrongSeed=lambda x: (x['WSeed'] == x['StrongSeed']).astype(int))
    ).copy()

    y = y.merge(ds.game_cities, on=['Season', 'DayNum', 'WTeamID', 'LTeamID', 'CRType'], how='left')
    y['ID'] = y["Season"].astype(str) + "_" + y["WTeamID"].astype(str) + "_" + y["LTeamID"].astype(str)
    
    y = _restack_winners_losers(y) 
    return _build_features_on_box_scores(y).rename(columns={'RankingDayNum': 'DayNum'})

    #results = _build_features_on_box_scores_v2(y)
    #return pd.concat([
    #    results.assign(TeamID=y['WTeamID'], Seed=y['WSeed'], _pred_label=1),
    #    results.assign(TeamID=y['LTeamID'], Seed=y['LSeed'], _pred_label=0)
    #]).drop(columns=['WTeamID', 'LTeamID', 'LSeed', 'WSeed'])

def prepare_x_dataset(ds: NCADataset):
    """ Method 1: Builds a simple training data X by merging Massey's Ordinal Ranks (Adds 8 Top Systems recommended by GPT), Game Scores, Team's Coaches and Seed. 
    ['Season', 'RankingDayNum', 'SystemName', 'TeamID', 'OrdinalRank']
    """
    # top_system_ranks = list(ds.massey_ordinals.SystemName.value_counts(ascending=False).head(100).index)
    massey_ordinals = ds.massey_ordinals[ds.massey_ordinals.SystemName.isin(SYS_BOOKIES)].copy()
    tourney = (
        massey_ordinals.rename(columns={'RankingDayNum': 'DayNum'})
        .pivot_table(
            index=["Season", "TeamID", "DayNum"],
            columns="SystemName",
            values="OrdinalRank",
        )
        .reset_index()
    )
    
    return (
        tourney
        # .merge(tourney_seed_slots,  on=['Season', 'TeamID'], how='left')
        .merge(ds.team_coaches,     on=['Season', 'TeamID'], how='left')
        .merge(ds.team_conferences, on=['Season', 'TeamID'], how='left')
        .merge(ds.teams[['TeamID', 'FirstD1Season', 'LastD1Season']], on=['TeamID'], how='left')
        .merge(ds.seasons, on=['Season'], how='left')
    )
    

In [515]:
# Preprocessing Qualitative Features 

from collections import namedtuple

cats = {}

CatLabel = namedtuple('CatLabel', ['tag2id', 'id2tag'])

def _build_encoder(vals: list):
    idx2tag = {idx: val for idx, val in enumerate(vals)}
    tag2idx = {val: idx for idx, val in enumerate(vals)}
    
    return CatLabel(
        tag2id=tag2idx,
        id2tag=idx2tag
    )
    
def _build_encoder_from_df(df: pd.DataFrame, string_col: str, id_col: str):
    
    idx2tag = df.set_index(id_col)[string_col].dropna().to_dict()
    tag2idx = df.set_index(string_col)[id_col].dropna().to_dict()
    
    return CatLabel(
        tag2id=tag2idx,
        id2tag=idx2tag
    )

if 'TeamID' not in cats:
    cats['TeamID'] = _build_encoder_from_df(male.teams, 'TeamName', 'TeamID')

def encode_categorical_fields(df: pd.DataFrame):
    global cats 
    
    cols = df.select_dtypes(include=['object', 'category', 'string']).columns
    if len(cols) == 0:
        print('Given dataframe has no obj/str column fields. Please pass another dataframe with obj/str dtype fields or refresh notebook.')
    
    try:
        for col in cols:
            if col not in ['ID']:
                print(f"Encoding obj field: {col}")
                vals = sorted(df[col].dropna().unique())
                cats[col] = _build_encoder(vals)
                df[col] = df[col].map(cats[col].tag2id)
            else:
                display(f'====== ! ! ! Skipping encoding `{col}` column')
        return df
    except Exception as e:
        display(df.head(10))
        raise e

def _map_game_round(day):
    if day <= 132:
        return 'Regular'
    elif 133 <= day <= 135:
        return 'First Four'
    elif 136 <= day <= 137:
        return 'Round of 64'
    elif 138 <= day <= 139:
        return 'Round of 32'
    elif 140 <= day <= 144:
        return 'Sweet 16'
    elif 145 <= day <= 146:
        return 'Elite 8'
    elif 152 <= day < 154:
        return 'Final Four'
    elif day >= 154:
        return 'Championship'
    else:
        return 'Unknown'
    
X = prepare_x_dataset(male)
y = prepare_y_dataset(male)
regular = load_regular_games(male)

display(f"Loaded X=({X.shape}), y=({y.shape})\n")

X = encode_categorical_fields(X)
y = encode_categorical_fields(y)

# y['WTeamName'] = y['WTeamID'].map(cats['TeamID'].id2tag)
# y['LTeamName'] = y['LTeamID'].map(cats['TeamID'].id2tag)

display(print(f"Total Kittens added to shelter: \n\n{len(cats)}\n== NCAA 2026 =="))
display(y.info(verbose=True))

AssertionError: 

In [504]:
y[y['Season']==2025].groupby('TeamID')['Seed'].describe()

KeyError: 'Column not found: Seed'

# Statistical Summary based on 2025

In [7]:
(
    X
    .loc[X['TeamID'] == 1196]
    .select_dtypes(exclude=['object', 'string'])
    .groupby(['Season', 'TeamID'])
    .agg(
        ['mean', 'std', 'min']
    )
    .pipe(lambda df: df.set_axis(
        ['_'.join(col) for col in df.columns], axis=1
    ))
    .reset_index()
)


,Season,TeamID,RankingDayNum_mean,RankingDayNum_std,RankingDayNum_min,DOK_mean,DOK_std,DOK_min,DOL_mean,DOL_std,...,RegionW_min,RegionX_mean,RegionX_std,RegionX_min,RegionY_mean,RegionY_std,RegionY_min,RegionZ_mean,RegionZ_std,RegionZ_min
0,2003,1196,85.266667,29.920043,37,NaN,NaN,NaN,4.000000,2.164007,...,2,5.0,0.0,5,2.0,0.0,2,6.0,0.0,6
1,2004,1196,72.736842,35.948107,13,NaN,NaN,NaN,27.857143,7.992682,...,1,4.0,0.0,4,1.0,0.0,1,3.0,0.0,3
2,2005,1196,77.615385,34.812764,16,NaN,NaN,NaN,44.000000,20.992306,...,0,0.0,0.0,0,0.0,0.0,0,4.0,0.0,4
3,2006,1196,71.590909,35.860620,13,8.555556,2.176052,5.0,7.200000,4.409387,...,1,3.0,0.0,3,3.0,0.0,3,5.0,0.0,5
4,2007,1196,88.909091,29.198310,37,2.866667,1.415802,1.0,8.181818,5.825433,...,2,5.0,0.0,5,2.0,0.0,2,6.0,0.0,6
5,2008,1196,73.382353,36.309290,1,41.850000,21.943168,1.0,44.181818,10.675375,...,2,1.0,0.0,1,5.0,0.0,5,6.0,0.0,6
6,2009,1196,76.843750,34.859265,6,36.833333,11.062656,13.0,38.764706,11.492005,...,2,5.0,0.0,5,2.0,0.0,2,6.0,0.0,6
7,2010,1196,74.212121,35.934592,8,41.578947,12.371745,17.0,48.214286,13.710493,...,2,5.0,0.0,5,2.0,0.0,2,6.0,0.0,6
8,2011,1196,69.142857,38.078492,0,32.000000,6.525647,17.0,27.000000,7.866870,...,2,6.0,0.0,6,6.0,0.0,6,2.0,0.0,2
9,2012,1196,70.102564,36.959178,7,11.500000,3.563837,5.0,31.214286,9.732657,...,2,1.0,0.0,1,5.0,0.0,5,6.0,0.0,6


In [10]:
(
    y
    .loc[
        (y['WTeamID'] == 1196) | \
        (y['LTeamID'] == 1196)  #\
        #(y['CRType'] == 'Regular')
    ]
    .select_dtypes(exclude=['object', 'string']) \
    .groupby(['Season', 'DayNum']) \
).corr()


WTeamID  WScore  LTeamID  LScore  WLoc  NumOT  WFGM  \
Season DayNum                                                                
2003   15     WTeamID      NaN     NaN      NaN     NaN   NaN    NaN   NaN   
              WScore       NaN     NaN      NaN     NaN   NaN    NaN   NaN   
              LTeamID      NaN     NaN      NaN     NaN   NaN    NaN   NaN   
              LScore       NaN     NaN      NaN     NaN   NaN    NaN   NaN   
              WLoc         NaN     NaN      NaN     NaN   NaN    NaN   NaN   
...                        ...     ...      ...     ...   ...    ...   ...   
2026   131    LStl         NaN     NaN      NaN     NaN   NaN    NaN   NaN   
              LBlk         NaN     NaN      NaN     NaN   NaN    NaN   NaN   
              LPF          NaN     NaN      NaN     NaN   NaN    NaN   NaN   
              CRType       NaN     NaN      NaN     NaN   NaN    NaN   NaN   
              CityID       NaN     NaN      NaN     NaN   NaN    NaN   NaN   

                       WFGA  WFGM3  WFGA3  ...  LFTA  LOR  LDR  LAst  LTO  \
Season DayNum                              ...                              
2003   15     WTeamID   NaN    NaN    NaN  ...   NaN  NaN  NaN   NaN  NaN   
              WScore    NaN    NaN    NaN  ...   NaN  NaN  NaN   NaN  NaN   
              LTeamID   NaN    NaN    NaN  ...   NaN  NaN  NaN   NaN  NaN   
              LScore    NaN    NaN    NaN  ...   NaN  NaN  NaN   NaN  NaN   
              WLoc      NaN    NaN    NaN  ...   NaN  NaN  NaN   NaN  NaN   
...                     ...    ...    ...  ...   ...  ...  ...   ...  ...   
2026   131    LStl      NaN    NaN    NaN  ...   NaN  NaN  NaN   NaN  NaN   
              LBlk      NaN    NaN    NaN  ...   NaN  NaN  NaN   NaN  NaN   
              LPF       NaN    NaN    NaN  ...   NaN  NaN  NaN   NaN  NaN   
              CRType    NaN    NaN    NaN  ...   NaN  NaN  NaN   NaN  NaN   
              CityID    NaN    NaN    NaN  ...   NaN  NaN  NaN   NaN  NaN   

                       LStl  LBlk  LPF  CRType  CityID  
Season DayNum                                           
2003   15     WTeamID   NaN   NaN  NaN     NaN     NaN  
              WScore    NaN   NaN  NaN     NaN     NaN  
              LTeamID   NaN   NaN  NaN     NaN     NaN  
              LScore    NaN   NaN  NaN     NaN     NaN  
              WLoc      NaN   NaN  NaN     NaN     NaN  
...                     ...   ...  ...     ...     ...  
2026   131    LStl      NaN   NaN  NaN     NaN     NaN  
              LBlk      NaN   NaN  NaN     NaN     NaN  
              LPF       NaN   NaN  NaN     NaN     NaN  
              CRType    NaN   NaN  NaN     NaN     NaN  
              CityID    NaN   NaN  NaN     NaN     NaN  

[28050 rows x 34 columns]

In [11]:
(
    y
    .groupby(['Season', 'WTeamID'])
    .size()
    .reset_index()
    .rename(columns={'WTeamID': 'TeamID', 0: 'TourneyWins'})
)

,Season,TeamID,TourneyWins
0,2003,1102,12
1,2003,1103,13
2,2003,1104,17
3,2003,1105,7
4,2003,1106,13
...,...,...,...
8333,2026,1477,9
8334,2026,1478,14
8335,2026,1479,15
8336,2026,1480,13


In [103]:
(
    y
    .groupby(['Season', 'LTeamID'])
    .size()
    .reset_index()
    .rename(columns={'LTeamID': 'TeamID', 0: 'TourneyLosses'})
)

,Season,TeamID,TourneyLosses
0,2003,1102,16
1,2003,1103,14
2,2003,1104,12
3,2003,1105,19
4,2003,1106,15
...,...,...,...
8341,2026,1477,21
8342,2026,1478,17
8343,2026,1479,17
8344,2026,1480,17


# Feature Engineering

## Massey Features

In [135]:
TARGET_YEAR = 2026
TARGET_DAY = 133
TARGET_RANK = 5

print(f"SEASON === {TARGET_YEAR} === MAX TOURNEY DAY ==== {male.massey_ordinals[male.massey_ordinals.Season == TARGET_YEAR].RankingDayNum.max()} ==== ")
display(
    (
        male.massey_ordinals[
            (male.massey_ordinals.Season == TARGET_YEAR ) &
            # (male.massey_ordinals.SystemName == 'SEL') &
            (male.massey_ordinals.RankingDayNum == TARGET_DAY ) 
            # (male.massey_ordinals.OrdinalRank == TARGET_RANK )
        ]# .TeamID.value_counts(normalize=True)
    )
)

# display(cats['TeamID'].id2tag.get(1222))

SEASON === 2026 === MAX TOURNEY DAY ==== 133 ==== 


,Season,RankingDayNum,SystemName,TeamID,OrdinalRank
5858066,2026,133,BIH,1101,233
5858067,2026,133,BIH,1102,346
5858068,2026,133,BIH,1103,45
5858069,2026,133,BIH,1104,14
5858070,2026,133,BIH,1105,266
...,...,...,...,...,...
5864996,2026,133,WLS,1477,318
5864997,2026,133,WLS,1478,303
5864998,2026,133,WLS,1479,287
5864999,2026,133,WLS,1480,293


In [271]:
regular.groupby('Season').get_group(2003)

,Season,TeamID,ID,reg_won_mean,reg_won_std,reg_won_median,reg_ptsfor_mean,reg_ptsfor_std,reg_ptsfor_median,reg_ptsagn_mean,...,reg_neteff_median,reg_fourfactors_mean,reg_fourfactors_std,reg_fourfactors_median,reg_poss_mean,reg_poss_std,reg_poss_median,WinRate,reg_numgames,reg_neteff_consistency
0,2003,1102,2003_1102_1117,1.0,NaN,1.0,57.0,NaN,57.0,52.0,...,9.491268,0.290877,NaN,0.290877,52.68,NaN,52.68,0.428571,28,NaN
1,2003,1102,2003_1102_1125,1.0,NaN,1.0,85.0,NaN,85.0,76.0,...,16.048502,0.441761,NaN,0.441761,56.08,NaN,56.08,0.428571,28,NaN
2,2003,1102,2003_1102_1161,1.0,NaN,1.0,80.0,NaN,80.0,75.0,...,8.069722,0.430102,NaN,0.430102,61.96,NaN,61.96,0.428571,28,NaN
3,2003,1102,2003_1102_1204,1.0,0.0,1.0,74.0,5.656854,74.0,48.0,...,46.205493,0.351189,0.007345,0.351189,55.72,1.753625,55.72,0.428571,28,0.034480
4,2003,1102,2003_1102_1298,1.0,NaN,1.0,56.0,NaN,56.0,43.0,...,23.431867,0.256607,NaN,0.256607,55.48,NaN,55.48,0.428571,28,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7413,2003,1464,2003_1464_1156,1.0,NaN,1.0,57.0,NaN,57.0,55.0,...,3.415301,0.287371,NaN,0.287371,58.56,NaN,58.56,0.259259,27,NaN
7414,2003,1464,2003_1464_1260,1.0,NaN,1.0,65.0,NaN,65.0,63.0,...,2.906977,0.257762,NaN,0.257762,68.80,NaN,68.80,0.259259,27,NaN
7415,2003,1464,2003_1464_1405,1.0,NaN,1.0,69.0,NaN,69.0,61.0,...,13.449899,0.282749,NaN,0.282749,59.48,NaN,59.48,0.259259,27,NaN
7416,2003,1464,2003_1464_1453,1.0,0.0,1.0,71.0,8.485281,71.0,68.5,...,4.270110,0.303930,0.023930,0.303930,60.36,4.072935,60.36,0.259259,27,0.262978


In [136]:
male.massey_ordinals.pivot_table(
    index=["Season", "RankingDayNum", "TeamID"],
    columns="SystemName",
    values="OrdinalRank",
)
#.columns to get the columns of all SystemName

SystemName                   7OT  ACU  ADE  AEI  AP  ARG  ATP  AUS  AWS  BAR  \
Season RankingDayNum TeamID                                                    
2003   35            1102    NaN  NaN  NaN  NaN NaN  NaN  NaN  NaN  NaN  NaN   
                     1103    NaN  NaN  NaN  NaN NaN  NaN  NaN  NaN  NaN  NaN   
                     1104    NaN  NaN  NaN  NaN NaN  NaN  NaN  NaN  NaN  NaN   
                     1105    NaN  NaN  NaN  NaN NaN  NaN  NaN  NaN  NaN  NaN   
                     1106    NaN  NaN  NaN  NaN NaN  NaN  NaN  NaN  NaN  NaN   
...                          ...  ...  ...  ...  ..  ...  ...  ...  ...  ...   
2026   133           1477    NaN  NaN  NaN  NaN NaN  NaN  NaN  NaN  NaN  NaN   
                     1478    NaN  NaN  NaN  NaN NaN  NaN  NaN  NaN  NaN  NaN   
                     1479    NaN  NaN  NaN  NaN NaN  NaN  NaN  NaN  NaN  NaN   
                     1480    NaN  NaN  NaN  NaN NaN  NaN  NaN  NaN  NaN  NaN   
                     1481    NaN  NaN  NaN  NaN NaN  NaN  NaN  NaN  NaN  NaN   

SystemName                   ...    WIL  WLK    WLS  WMR  WMV  WOB  WOL  WTE  \
Season RankingDayNum TeamID  ...                                               
2003   35            1102    ...    NaN  NaN    NaN  NaN  NaN  NaN  NaN  NaN   
                     1103    ...    NaN  NaN    NaN  NaN  NaN  NaN  NaN  NaN   
                     1104    ...    NaN  NaN    NaN  NaN  NaN  NaN  NaN  NaN   
                     1105    ...    NaN  NaN    NaN  NaN  NaN  NaN  NaN  NaN   
                     1106    ...    NaN  NaN    NaN  NaN  NaN  NaN  NaN  NaN   
...                          ...    ...  ...    ...  ...  ...  ...  ...  ...   
2026   133           1477    ...  303.0  NaN  318.0  NaN  NaN  NaN  NaN  NaN   
                     1478    ...  314.0  NaN  303.0  NaN  NaN  NaN  NaN  NaN   
                     1479    ...  307.0  NaN  287.0  NaN  NaN  NaN  NaN  NaN   
                     1480    ...  268.0  NaN  293.0  NaN  NaN  NaN  NaN  NaN   
                     1481    ...  325.0  NaN  308.0  NaN  NaN  NaN  NaN  NaN   

SystemName                   YAG  ZAM  
Season RankingDayNum TeamID            
2003   35            1102    NaN  NaN  
                     1103    NaN  NaN  
                     1104    NaN  NaN  
                     1105    NaN  NaN  
                     1106    NaN  NaN  
...                          ...  ...  
2026   133           1477    NaN  NaN  
                     1478    NaN  NaN  
                     1479    NaN  NaN  
                     1480    NaN  NaN  
                     1481    NaN  NaN  

[270746 rows x 197 columns]

In [ ]:
# To get the top 50 Systems based on its data entry counts 

male.massey_ordinals[male.massey_ordinals['OrdinalRank'] <= 50]\
    .merge(male.teams, on='TeamID')\
    [['Season', 'TeamID', 'SystemName', 'OrdinalRank']]\
    .sort_values('Season', ascending=False)\
    [['SystemName']].value_counts(ascending=False).reset_index()['SystemName']

array(['MOR', 'POM', 'DOK', 'SAG', 'MAS', 'WLK', 'WIL', 'DOL', 'PGH',
       'COL', 'KPK', 'BIH', 'DUN', 'WOL', 'RTH', 'PIG', 'RT', 'DII',
       'DCI', 'TRP', 'RTP', 'NOL', 'EBP', 'RPI', 'SEL', 'TRK', 'USA',
       'AP', 'BWE', 'BBT', 'BOB', 'SE', 'CNG', 'LOG', 'HAS', 'WOB', '7OT',
       'SFX', 'DES', 'MB', 'FAS', 'STH', 'SPR', 'TPR', 'JNG', 'ARG',
       'RTR', 'LMC', 'REW', 'INC', 'KRA', 'SMS', 'LYN', 'GRN', 'NET',
       'DAV', 'DC', 'SPW', 'BNZ', 'EMK', 'BUR', 'RTB', 'TMR', 'KLK',
       'D1A', 'SP', 'PIR', 'TW', 'FSH', 'ESR', 'ACU', 'OMY', 'JJK', 'MMG',
       'SIM', 'CRO', 'RMS', 'CJB', 'SRS', 'REI', 'JEN', 'BLS', 'MSX',
       'STY', 'SMN', 'MGS', 'TSR', 'PAC', 'LEF', 'HKB', 'DC2', 'ADE',
       'ENT', 'WMV', 'KPI', 'DP', 'BAR', 'PTS', 'ISR', 'ECK', 'STR',
       'DWH', 'SAU', 'MKV', 'SGR', 'KCX', 'HER', 'ZAM', 'ROH', 'BMN',
       'CBR', 'BNT', 'BPI', 'YAG', 'WMR', 'CPR', 'CPA', 'FMG', 'STF',
       'KMV', 'SAP', 'OMN', 'BD', 'WLS', 'KOS', 'RME', 'TRX', 'SCR',
       'PKL', '

In [247]:
(
    male.massey_ordinals[
        (male.massey_ordinals['Season'] <= 2025)
    ].rename(columns={'RankingDayNum': 'DayNum'})
    .sort_values('DayNum')
    .groupby(['Season', 'SystemName', 'TeamID'], as_index=False)
    .last()  # most recent rank before tourney
    .pivot_table(
        index=["Season", "DayNum", "TeamID"],
        columns="SystemName",
        values="OrdinalRank",
    )
    .merge(
        y,
        on=['Season', 'TeamID', 'DayNum'],
        how='left'
    )
)


,Season,TeamID,DayNum,7OT,ACU,ADE,AP,ARG,ATP,AUS,...,CRType,GameRound,Slot,EarlyDayNum,LateDayNum,StrongSeed,CityID,ID,Seed,_pred_label
0,2003,1158,37,NaN,NaN,NaN,25.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2003,1278,37,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2003,1278,44,NaN,NaN,NaN,25.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2003,1314,51,NaN,NaN,NaN,22.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2003,1409,51,NaN,NaN,NaN,20.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
23762,2025,1476,133,310.0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
23763,2025,1477,133,338.0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
23764,2025,1478,133,329.0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
23765,2025,1479,133,334.0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import hmean
from typing import Optional

def build_massey_features(
    df_massey: pd.DataFrame,
    season_min: int = 1985,
    season_max: int = 2025,
    ranking_day: int = 128,
    systems_to_use: Optional[list] = None,
) -> pd.DataFrame:
    """
    Compute team‑season Massey ordinal features from MMasseyOrdinals.

    Parameters
    ----------
    df_massey : pd.DataFrame
        Kaggle MMasseyOrdinals (or MMasseyOrdinalsM/MMasseyOrdinalsW).
    season_min : int
        Minimum season to keep.
    ranking_day : int
        DayNum to use as "latest" pre‑tournament rank (e.g., 128).
    systems_to_use : list of str, optional
        List of SystemName values to keep; if None, use all systems that appear
        in every season in the range season_min–max(Season).

    Returns
    -------
    pd.DataFrame
        Index: (Season, TeamID); columns: Massey‑derived features.
    """
    df = df_massey.copy()

    # Filter by season
    df = df[
        (df["Season"] >= season_min) & 
        (df["Season"] < season_max)
    ]

    # Determine common systems across all seasons (if requested)
    if systems_to_use is None:
        season_systems = df.groupby("Season")["SystemName"].apply(set)
        common_systems = set.intersection(*season_systems)
        systems_to_use = sorted(common_systems)

    df = df[df["SystemName"].isin(systems_to_use)]

    # 1) Take latest rank per (Season, TeamID, SystemName)
    df_latest = (
        df
        [df["RankingDayNum"] <= ranking_day]
        .sort_values("RankingDayNum", ascending=False)
        .groupby(["Season", "TeamID", "SystemName"])
        .first()  # first = last in this sorted order
        .reset_index()
    )

    # Pivot: SystemName → rank columns
    team_sys = df_latest.pivot(
        index=["Season", "TeamID"],
        columns="SystemName",
        values="OrdinalRank"
    ).reset_index()

    # 2) Compute multi‑system aggregates per (Season, TeamID)
    sys_cols = [c for c in team_sys.columns if c not in ["Season", "TeamID"]]

    features = team_sys[["Season", "TeamID"]].copy()

    # 2a) Basic stats across systems
    features["massey_rank_mean"] = team_sys[sys_cols].mean(axis=1)
    features["massey_rank_median"] = team_sys[sys_cols].median(axis=1)
    features["massey_rank_stdev"] = team_sys[sys_cols].std(axis=1)
    features["massey_rank_min"] = team_sys[sys_cols].min(axis=1)
    features["massey_rank_max"] = team_sys[sys_cols].max(axis=1)
    features["massey_rank_range"] = features["massey_rank_max"] - features["massey_rank_min"]

    # 2b) Inverse‑rank “strength”
    inv_ranks = 1.0 / team_sys[sys_cols]
    features["massey_inv_rank_sum"] = inv_ranks.sum(axis=1)
    features["massey_inv_rank_mean"] = inv_ranks.mean(axis=1)

    # 2c) Harmonic mean of ranks (across systems)
    features["massey_rank_hmean"] = np.nan
    mask = team_sys[sys_cols].all(axis=1, skipna=True)
    if mask.any():
        # hmean works on array of positive values
        hmean_vals = np.apply_along_axis(
            lambda row: hmean(row[row > 0]), 1, team_sys[sys_cols].values
        )
        features.loc[mask, "massey_rank_hmean"] = hmean_vals

    # 3) Momentum / change indicators (over last N days)
    # Keep a wide table of last‑available rank per (Season, TeamID, SystemName)
    # and then subtract from the latest‑at‑ranking_day
    df_history = (
        df[df["RankingDayNum"] <= ranking_day]
        .sort_values("RankingDayNum", ascending=False)
        .groupby(["Season", "TeamID", "SystemName"])
        .nth(0)  # latest rank for each team/system
        .reset_index()
    )
    df_prev = (
        df[df["RankingDayNum"] <= ranking_day - 10]
        .sort_values("RankingDayNum", ascending=False)
        .groupby(["Season", "TeamID", "SystemName"])
        .nth(0)  # rank ~10 days earlier
        .reset_index()
    )

    df_change = df_latest[["Season", "TeamID", "SystemName", "OrdinalRank"]].merge(
        df_prev[["Season", "TeamID", "SystemName", "OrdinalRank"]],
        on=["Season", "TeamID", "SystemName"],
        suffixes=("", "_prev"),
        how="left"
    )

    df_change["rank_drop"] = df_change["OrdinalRank"] - df_change["Rank_prev"]

    # Global drop indicators (across all systems for this team)
    drop_means = (
        df_change
        .groupby(["Season", "TeamID"])["rank_drop"]
        .mean()
        .rename("massey_rank_drop_mean")
    )
    drop_stds = (
        df_change
        .groupby(["Season", "TeamID"])["rank_drop"]
        .std()
        .rename("massey_rank_drop_stdev")
    )

    features = features.merge(drop_means, on=["Season", "TeamID"], how="left")
    features = features.merge(drop_stds, on=["Season", "TeamID"], how="left")

    # Optional: also add “true‑odds‑style” base metrics (e.g., logistic‑type win prob vs avg opponent)
    # (You can refine this later using a fitted logistic on historical games.)
    # features["massey_baseline_logit"] = - (features["massey_rank_mean"] - 50.) * 0.1

    return features.reset_index(drop=True)


In [353]:
import pandas as pd
import numpy as np

def predict_outcome_from_bookie(
    df: pd.DataFrame,
    sys_cols: list[str] = SYS_BOOKIES,
    daynum_cutoff: int = 128,
) -> pd.DataFrame:
    """
    Given a DataFrame with bookie‑style system scores per (Season, TeamID, DayNum),
    compute aggregated stats and then convert to probability‑like scores.

    Parameters
    ----------
    df : pd.DataFrame
        With columns: Season, TeamID, DayNum, and sys_cols.
    sys_cols : list of str
        List of bookie‑style system columns (e.g., DOK, MAS, POM, SAG, ...).
    daynum_cutoff : int
        Only keep DayNum <= daynum_cutoff (e.g., last pre‑tournament day).

    Returns
    -------
    pd.DataFrame
        Index: (Season, TeamID, DayNum); additional columns:
            - mean/max/min of bookie scores per day
            - logistic‑style "prob" (example parameterization you can tune).
    """
    df_use = df[df["DayNum"] <= daynum_cutoff].copy()

    # Step 1: simple aggregates per (Season, TeamID, DayNum)
    X = (
        df_use
        [["Season", "TeamID", "DayNum"] + sys_cols]
        .groupby(["Season", "TeamID", "DayNum"])
        .agg(["mean", "max", "min"])  # this is your current line
        .round(3)
    )

    # Flatten columns: e.g., "MAS", "POM" → "MAS_mean", "MAS_max", "MAS_min"
    X.columns = ["_".join(col).strip() for col in X.columns]

    # Step 2: convert to “probability‑like” scores

    # For each system, if values are **ranks** (lower = better), invert:
    #   strength = 1 / (rank + 1e‑8)  → higher strength = better team
    # Then aggregate across systems and squash into [0,1] via sigmoid.

    sys_base = [c for c in X.columns if any(s in c for s in sys_cols)]
    mean_cols = [c for c in sys_base if c.endswith("_mean")]

    # For each row: transformed strength = sum(1/rank) across systems
    inv_strength = pd.DataFrame(
        np.zeros(len(X)), index=X.index, columns=["bookie_strength"]
    )
    for sys in sys_cols:
        mean_col = f"{sys}_mean"
        if mean_col in X.columns:
            ranks = X[mean_col]  # lower rank = better
            # avoid 0 / Inf by adding small offset
            inv_strength["bookie_strength"] += 1.0 / (ranks + 1e-8)

    # Logistic‑style “win probability vs average opponent”
    # (you can tune coeffs or refit on historical data)
    prob_logit = 5.0 - inv_strength["bookie_strength"]  # example: worse = lower prob
    X["bookie_prob"] = 1.0 / (1.0 + np.exp(prob_logit))

    # Optional: also keep raw mean‑max‑min per system
    X["bookie_mean_all"] = X[mean_cols].mean(axis=1)
    X["bookie_max_all"] = X[mean_cols].max(axis=1)
    X["bookie_min_all"] = X[mean_cols].min(axis=1)

    X = X.reset_index(drop=False)

    return X

predict_outcome_from_bookie(X)

,Season,TeamID,DayNum,MOR_mean,MOR_max,MOR_min,POM_mean,POM_max,POM_min,DOK_mean,...,WIL_mean,WIL_max,WIL_min,DOL_mean,DOL_max,DOL_min,bookie_prob,bookie_mean_all,bookie_max_all,bookie_min_all
0,2003,1102,37,110.0,110.0,110.0,104.0,104.0,104.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,104.200000,114.0,90.0
1,2003,1102,44,106.0,106.0,106.0,84.0,84.0,84.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,75.200000,106.0,52.0
2,2003,1102,51,127.0,127.0,127.0,126.0,126.0,126.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,103.600000,127.0,52.0
3,2003,1102,57,124.0,124.0,124.0,156.0,156.0,156.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,133.000000,156.0,121.0
4,2003,1102,65,119.0,119.0,119.0,134.0,134.0,134.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,122.600000,134.0,116.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
185999,2026,1481,100,329.0,329.0,329.0,339.0,339.0,339.0,333.0,...,330.0,330.0,330.0,322.0,322.0,322.0,NaN,331.714286,352.0,317.0
186000,2026,1481,107,318.0,318.0,318.0,332.0,332.0,332.0,333.0,...,325.0,325.0,325.0,316.0,316.0,316.0,NaN,326.714286,351.0,312.0
186001,2026,1481,114,311.0,311.0,311.0,329.0,329.0,329.0,328.0,...,322.0,322.0,322.0,306.0,306.0,306.0,NaN,320.428571,343.0,304.0
186002,2026,1481,121,313.0,313.0,313.0,329.0,329.0,329.0,327.0,...,327.0,327.0,327.0,316.0,316.0,316.0,NaN,324.428571,346.0,313.0


### Normalizing Bookies

In [624]:
v = score[[col for col in score.columns if col not in ('TeamID')]].values
inv = 1.0 / (v + 1e-2)
arr_norm = inv / inv.sum(axis=1, keepdims=True)

In [630]:
import numpy as np

In [632]:
(v - v.min(axis=1, keepdims=True)) / (v.max(axis=1, keepdims=True) - v.min(axis=1, keepdims=True) + 1e-8)

array([[1.        , 0.        , 0.20754717, 0.10377358, 0.23584906,
        0.28301887, 0.20754717, 0.20754717, 0.11320755, 0.16981132,
        0.23584906, 0.05660377, 0.18867925, 0.11320755, 0.19811321,
        0.12264151, 0.17924528, 0.1509434 , 0.14150943, 0.06603774,
        0.32075472, 0.13207547, 0.18867925, 0.        , 0.17924528,
        0.40566038, 0.22641509, 0.1509434 , 0.16981132, 0.18867925,
        0.24528302, 0.17924528, 0.17924528, 0.17924528, 0.13207547,
        0.19811321]])

In [673]:
season_sys = grp.sort_values(by=['TeamID']).groupby('SystemName').get_group('AP').sort_values(by='Rank').copy()
season_sys['Proba'] = 1/(season_sys['Rank'] + 1e-8)

In [680]:
bookies = {}

In [ ]:
bookies[2003].setdefault({}, )

In [692]:
bookies.setdefault(2003, {}).update(season_sys.set_index('TeamID')['Proba'].to_dict())

In [697]:
bookies

{2003: {1112: 0.9999999900000002,
  1163: 0.045454545433884296,
  1166: 0.07692307686390532,
  1173: 0.020408163261141192,
  1181: 0.0999999999,
  1196: 0.07142857137755101,
  1208: 0.14285714265306124,
  1228: 0.1111111109876543,
  1242: 0.1999999996,
  1246: 0.3333333322222222,
  1257: 0.249999999375,
  1266: 0.05263157891966759,
  1268: 0.04166666664930555,
  1272: 0.0322580645057232,
  1280: 0.12499999984374999,
  1281: 0.04761904759637188,
  1323: 0.09090909082644627,
  1328: 0.0624999999609375,
  1338: 0.4999999975,
  1390: 0.02222222221728395,
  1393: 0.05555555552469135,
  1400: 0.06666666662222222,
  1448: 0.16666666638888888,
  1458: 0.049999999975,
  1462: 0.08333333326388888,
  1102: 0.006410256409845496,
  1103: 0.006211180123837814,
  1104: 0.03571428570153061,
  1105: 0.003344481605239315,
  1106: 0.003906249999847412,
  1107: 0.003267973856102354,
  1108: 0.0035211267604393966,
  1110: 0.006329113923650056,
  1111: 0.006578947367988227,
  1113: 0.019230769227071006,
  1

In [699]:
import pandas as pd

# Load the official template — this defines the exact 132,133 rows you need
sample_sub = pd.read_csv('/kaggle/input/competitions/march-machine-learning-mania-2026/SampleSubmissionStage2.csv')

print(f"Required rows: {len(sample_sub)}")  # Should be 132,133

# Parse IDs to get Team1, Team2
sample_sub[['Season', 'Team1ID', 'Team2ID']] = (
    sample_sub['ID'].str.split('_', expand=True).astype(int)
)

# === Your model prediction logic here ===
# For each row, compute the predicted probability that Team1ID beats Team2ID
# e.g. sample_sub['Pred'] = model.predict_proba(X_pred)[:, 1]

# If your model can't score a matchup (missing features), default to 0.5
sample_sub['Pred'] = sample_sub['Pred'].fillna(0.5)

# Final submission — keep ONLY ID and Pred, same order as template
submission = sample_sub[['ID', 'Pred']].copy()

# Sanity checks
assert len(submission) == 132133, f"Wrong row count: {len(submission)}"
assert submission['Pred'].notna().all(), "NaN predictions found!"
assert (submission['Pred'] >= 0).all() and (submission['Pred'] <= 1).all(), "Preds out of range!"

submission.to_csv('/kaggle/working/submission.csv', index=False)
print(f"Submission saved: {len(submission)} rows ✓")

Required rows: 132133
Submission saved: 132133 rows ✓


In [701]:
submission

,ID,Pred
0,2026_1101_1102,0.5
1,2026_1101_1103,0.5
2,2026_1101_1104,0.5
3,2026_1101_1105,0.5
4,2026_1101_1106,0.5
...,...,...
132128,2026_3478_3480,0.5
132129,2026_3478_3481,0.5
132130,2026_3479_3480,0.5
132131,2026_3479_3481,0.5


In [452]:
(
    1/(
        X[X["Season"] < 2025][SYS_BOOKIES]
        .fillna(method="ffill", axis=1)        # across columns
    ).values.sum(axis=-1).reshape(-1, 1) * (
        X[X["Season"] < 2025][SYS_BOOKIES]
        .fillna(method="ffill", axis=1)        # across columns
    ).values
)#.sum(axis=-1).shape

array([[0.13237064, 0.12515042, 0.12515042, ..., 0.12394705, 0.12394705,
        0.12394705],
       [0.18794326, 0.14893617, 0.14893617, ..., 0.09219858, 0.09219858,
        0.09219858],
       [0.1697861 , 0.1684492 , 0.1684492 , ..., 0.06951872, 0.06951872,
        0.06951872],
       ...,
       [0.11429725, 0.12358643, 0.12722132, ..., 0.12075929, 0.13045234,
        0.12560582],
       [0.11460033, 0.1223491 , 0.12520392, ..., 0.12316476, 0.13132137,
        0.12683524],
       [0.11460033, 0.1223491 , 0.12561175, ..., 0.1223491 , 0.1317292 ,
        0.12642741]])

In [383]:
X[['Season', 'DayNum', 'TeamID']+SYS_BOOKIES]

,Season,DayNum,TeamID,MOR,POM,DOK,SAG,MAS,WLK,WIL,DOL
0,2003,37,1102,110.0,104.0,NaN,114.0,90.0,103.0,NaN,NaN
1,2003,44,1102,106.0,84.0,NaN,78.0,56.0,52.0,NaN,NaN
2,2003,51,1102,127.0,126.0,NaN,113.0,100.0,52.0,NaN,NaN
3,2003,57,1102,124.0,156.0,NaN,127.0,137.0,121.0,NaN,NaN
4,2003,65,1102,119.0,134.0,NaN,124.0,120.0,116.0,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
195994,2026,107,1481,318.0,332.0,333.0,NaN,351.0,312.0,325.0,316.0
195995,2026,114,1481,311.0,329.0,328.0,NaN,343.0,304.0,322.0,306.0
195996,2026,121,1481,313.0,329.0,327.0,NaN,346.0,313.0,327.0,316.0
195997,2026,128,1481,313.0,328.0,328.0,NaN,345.0,315.0,325.0,318.0


In [517]:
y[
    (y['Season'] == 2003) &
    (y['DayNum'] > 134)
]

,ID,Season,TeamID,DayNum,Won,PtsFor,PtsAgn,Seed,FGM,FGA,...,TOVRate,AstTORatio,AstRate,StlRate,BlkRate,OffEff,DefEff,NetEff,ScoreMargin,FourFactors
7,2003_1112_1436,2003,1112,136,1,80,51,Z01,31,66,...,0.207361,1.375,0.709677,0.126326,0.109375,103.680664,66.096423,37.584241,29,0.248528
8,2003_1112_1436,2003,1112,136,1,80,51,Z01,31,66,...,0.207361,1.375,0.709677,0.126326,0.109375,103.680664,66.096423,37.584241,29,0.248528
9,2003_1112_1436,2003,1112,136,1,80,51,Z01,31,66,...,0.207361,1.375,0.709677,0.126326,0.109375,103.680664,66.096423,37.584241,29,0.248528
10,2003_1112_1436,2003,1112,136,1,80,51,Z01,31,66,...,0.207361,1.375,0.709677,0.126326,0.109375,103.680664,66.096423,37.584241,29,0.248528
11,2003_1112_1436,2003,1112,136,1,80,51,Z01,31,66,...,0.207361,1.375,0.709677,0.126326,0.109375,103.680664,66.096423,37.584241,29,0.248528
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9203,2003_1393_1242,2003,1242,154,0,78,81,Z02,31,71,...,0.236220,1.000,0.580645,0.109489,0.063492,102.362205,106.299213,-3.937008,-3,0.292201
9204,2003_1393_1242,2003,1242,154,0,78,81,Z02,31,71,...,0.236220,1.000,0.580645,0.109489,0.063492,102.362205,106.299213,-3.937008,-3,0.292201
9205,2003_1393_1242,2003,1242,154,0,78,81,Z02,31,71,...,0.236220,1.000,0.580645,0.109489,0.063492,102.362205,106.299213,-3.937008,-3,0.292201
9206,2003_1393_1242,2003,1242,154,0,78,81,Z02,31,71,...,0.236220,1.000,0.580645,0.109489,0.063492,102.362205,106.299213,-3.937008,-3,0.292201


In [535]:
massey = (
    male.massey_ordinals[
        (male.massey_ordinals['Season'] < 2025) & 
        (male.massey_ordinals['RankingDayNum'] >= 133)
    ]
).rename(columns={'RankingDayNum': 'DayNum', 'OrdinalRank': 'Rank'})

In [ ]:
c=0
BOOKIES = {}

for (season, day), grp in massey.groupby(['Season', 'DayNum']):
    x = grp.pivot_table(
        index=["Season", "TeamID", "DayNum" ],
        columns="SystemName",
        values="Rank",
    ).reset_index().fillna(method='ffill', axis=1)
    
    for team, score in x[[col for col in x.columns if col not in ('Season', 'DayNum')]].groupby('TeamID'):
        print(f"==team:\t\t{team}")
        print(f"==bookies:")
        display(score)
        
        market_ = score.to_dict(orient='records')
        
        if not isinstance(market_, list): raise ValueError
        if isinstance(market_, list) and len(market_) > 1:
            raise ValueError 
        
        market = market_[0]
        team_true_result = y[
            (y['Season'] == season) & 
            (y['TeamID'] == int(market.get('TeamID', None))) 
        ].drop_duplicates()
        
        if team_true_result.empty:
            print('Skipping entry -- could not find matching true outcomes')
        else: 
            print(f"==Won:\t\t{team_true_result["Won"][0]}")
            print("==true outcome:")
            display(team_true_result)
        break
    
    if c == 1:
        break
    
    c+=1
    

KeyError: 'DayNum'

In [ ]:
c=0
bookies = {}
bookies_ranks = {}

for (season, day), grp in massey.groupby(['Season', 'DayNum']):
    print(f"=={season}\n=={day}")
    market = []
    for book, board in grp.groupby("SystemName"):
        print(f"==bookie: {book}")
        
        zdf = board.copy()
        zdf['Proba'] = 1 / (board.sort_values(by="Rank")['Rank'] + 1e-8)
        bookies.setdefault(int(season), {}).update(zdf.set_index('TeamID')['Proba'].to_dict())
        bookies_ranks \
            .setdefault(int(season), {}) \
                .setdefault(1)
            .update(zdf.set_index('Rank')['TeamID'].to_dict())
        
    if c == 2:
        break
    
    c+=1

==2003
==133
==bookie: AP
==bookie: ARG
==bookie: BIH
==bookie: BOB
==bookie: BRZ
==bookie: COL
==bookie: DOL
==bookie: DUN
==bookie: DWH
==bookie: ECK
==bookie: ENT
==bookie: ERD
==bookie: GRN
==bookie: GRS
==bookie: HER
==bookie: HOL
==bookie: IMS
==bookie: MAS
==bookie: MKV
==bookie: MOR
==bookie: POM
==bookie: RPI
==bookie: RTH
==bookie: SAG
==bookie: SAU
==bookie: SE
==bookie: SEL
==bookie: STR
==bookie: TSR
==bookie: USA
==bookie: WLK
==bookie: WOB
==bookie: WOL
==bookie: WTE
==2004
==133
==bookie: AP
==bookie: ARG
==bookie: BD
==bookie: BIH
==bookie: BOB
==bookie: CNG
==bookie: COL
==bookie: DES
==bookie: DOL
==bookie: DWH
==bookie: ECK
==bookie: ENT
==bookie: GRN
==bookie: GRS
==bookie: HER
==bookie: JON
==bookie: LYN
==bookie: MAS
==bookie: MGY
==bookie: MKV
==bookie: MOR
==bookie: NOR
==bookie: POM
==bookie: REI
==bookie: RM
==bookie: RPI
==bookie: RTH
==bookie: SAG
==bookie: SAU
==bookie: SEL
==bookie: SIM
==bookie: TSR
==bookie: USA
==bookie: WLK
==bookie: WOB
==bookie: WOL

# Querying & Viewing Dataset

In [189]:
# To view NCAA tournament for 2025 winners correlation

y[
    (y.Season == 2025) & \
    (y.CRType == cats['CRType'].tag2id.get('NCAA')) & \
    (y.DayNum >= 152)
].select_dtypes(exclude=['object', 'string']).corr() 

,Season,DayNum,WTeamID,WScore,LTeamID,LScore,WLoc,NumOT,WFGM,WFGA,...,LFTA,LOR,LDR,LAst,LTO,LStl,LBlk,LPF,CRType,CityID
Season,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
DayNum,NaN,1.000000,-0.500000,-0.773099,0.804285,-0.802955,NaN,NaN,-8.660254e-01,-5.000000e-01,...,-1.000000,0.970725,0.944911,-0.944911,-0.327327,-0.755929,0.944911,0.000000,NaN,NaN
WTeamID,NaN,-0.500000,1.000000,-0.162758,0.112487,-0.114708,NaN,NaN,0.000000e+00,1.000000e+00,...,0.500000,-0.277350,-0.755929,0.755929,-0.654654,-0.188982,-0.188982,-0.866025,NaN,NaN
WScore,NaN,-0.773099,-0.162758,1.000000,-0.998712,0.998823,NaN,NaN,9.866661e-01,-1.627577e-01,...,0.773099,-0.902817,-0.522891,0.522891,0.852399,0.999645,-0.938129,0.634285,NaN,NaN
LTeamID,NaN,0.804285,0.112487,-0.998712,1.000000,-0.999998,NaN,NaN,-9.936531e-01,1.124874e-01,...,-0.804285,0.923473,0.565466,-0.565466,-0.824771,-0.997006,0.954490,-0.594244,NaN,NaN
LScore,NaN,-0.802955,-0.114708,0.998823,-0.999998,1.000000,NaN,NaN,9.933993e-01,-1.147079e-01,...,0.802955,-0.922613,-0.563621,0.563621,0.826033,0.997176,-0.953821,0.596040,NaN,NaN
WLoc,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
NumOT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
WFGM,NaN,-0.866025,0.000000,0.986666,-0.993653,0.993399,NaN,NaN,1.000000e+00,-3.845925e-16,...,0.866025,-0.960769,-0.654654,0.654654,0.755929,0.981981,-0.981981,0.500000,NaN,NaN
WFGA,NaN,-0.500000,1.000000,-0.162758,0.112487,-0.114708,NaN,NaN,-3.845925e-16,1.000000e+00,...,0.500000,-0.277350,-0.755929,0.755929,-0.654654,-0.188982,-0.188982,-0.866025,NaN,NaN


In [140]:
y.corr()

,Season,DayNum,WTeamID,WScore,LTeamID,LScore,WLoc,NumOT,WFGM,WFGA,...,LFTA,LOR,LDR,LAst,LTO,LStl,LBlk,LPF,CRType,CityID
Season,1.000000,-0.045420,0.019898,0.132778,0.018326,0.133038,-0.003144,-0.006053,0.139265,0.161784,...,-0.031568,-0.284573,0.039515,-0.011476,-0.260250,-0.051938,0.017044,-0.164447,0.014607,0.043523
DayNum,-0.045420,1.000000,-0.001310,-0.088653,0.004451,0.038707,-0.034419,0.027450,-0.084164,-0.087929,...,-0.001150,-0.019851,-0.002144,0.021668,-0.168401,-0.045424,0.020087,-0.086811,-0.199730,-0.023270
WTeamID,0.019898,-0.001310,1.000000,-0.006734,0.007337,-0.010211,0.004892,-0.005506,-0.005652,0.000054,...,-0.003222,-0.022167,-0.002613,-0.004166,-0.015403,-0.014819,0.002864,0.008151,0.001102,0.117590
WScore,0.132778,-0.088653,-0.006734,1.000000,0.002336,0.643774,0.046581,0.133068,0.819076,0.491710,...,0.262073,0.038896,-0.122697,0.280885,0.048402,0.013813,-0.053105,0.205775,0.004633,-0.000694
LTeamID,0.018326,0.004451,0.007337,0.002336,1.000000,0.016763,0.011990,0.004743,0.005201,0.011657,...,0.008602,0.003582,0.006557,0.011657,-0.024754,-0.014719,0.003488,-0.012116,-0.009142,0.064799
LScore,0.133038,0.038707,-0.010211,0.643774,0.016763,1.000000,-0.048465,0.291080,0.435122,0.317566,...,0.340923,0.090107,0.090935,0.504304,-0.124685,0.069811,0.040587,0.298974,0.002146,0.000640
WLoc,-0.003144,-0.034419,0.004892,0.046581,0.011990,-0.048465,1.000000,-0.025352,0.014645,0.024421,...,-0.071134,-0.008340,-0.000124,-0.089236,0.054571,-0.023686,-0.075637,0.084654,-0.207801,0.025167
NumOT,-0.006053,0.027450,-0.005506,0.133068,0.004743,0.291080,-0.025352,1.000000,0.071188,0.215186,...,0.152287,0.093338,0.230757,0.141936,0.041222,0.069210,0.084493,0.147777,0.001536,0.002841
WFGM,0.139265,-0.084164,-0.005652,0.819076,0.005201,0.435122,0.014645,0.071188,1.000000,0.633283,...,0.106982,-0.014803,-0.138898,0.214536,0.041218,-0.023100,-0.054904,-0.207891,-0.001951,-0.003053
WFGA,0.161784,-0.087929,0.000054,0.491710,0.011657,0.317566,0.024421,0.215186,0.633283,1.000000,...,0.117269,-0.006018,0.412396,0.151928,0.154096,-0.118021,0.228235,-0.166514,-0.001431,0.002770


In [16]:
X[X.TeamID == 1196].RankingDayNum.map(_map_game_round).value_counts()

RankingDayNum
Regular       2466
First Four     108
Name: count, dtype: int64

In [288]:
X.columnsa

,Season,TeamID,DayNum,DOK,DOL,MAS,MOR,POM,SAG,WIL,...,LastDayNum,CoachName,ConfAbbrev,FirstD1Season,LastD1Season,DayZero,RegionW,RegionX,RegionY,RegionZ
0,2003,1102,37,NaN,NaN,90.0,110.0,104.0,114.0,NaN,...,154,533,22.0,1985,2026,16,2,5,2,6
1,2003,1102,44,NaN,NaN,56.0,106.0,84.0,78.0,NaN,...,154,533,22.0,1985,2026,16,2,5,2,6
2,2003,1102,51,NaN,NaN,100.0,127.0,126.0,113.0,NaN,...,154,533,22.0,1985,2026,16,2,5,2,6
3,2003,1102,57,NaN,NaN,137.0,124.0,156.0,127.0,NaN,...,154,533,22.0,1985,2026,16,2,5,2,6
4,2003,1102,65,NaN,NaN,120.0,119.0,134.0,124.0,NaN,...,154,533,22.0,1985,2026,16,2,5,2,6
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
195994,2026,1481,107,333.0,316.0,351.0,318.0,332.0,NaN,325.0,...,132,1017,23.0,2026,2026,15,2,5,2,6
195995,2026,1481,114,328.0,306.0,343.0,311.0,329.0,NaN,322.0,...,132,1017,23.0,2026,2026,15,2,5,2,6
195996,2026,1481,121,327.0,316.0,346.0,313.0,329.0,NaN,327.0,...,132,1017,23.0,2026,2026,15,2,5,2,6
195997,2026,1481,128,328.0,318.0,345.0,313.0,328.0,NaN,325.0,...,132,1017,23.0,2026,2026,15,2,5,2,6


a) From ordinal ranks alone (simple heuristics)
Rank difference → logit:

Transform a rank difference 
Δ
r
=
r
1
−
r
2
Δr=r 
1
 −r 
2
  into win probability via a logistic‑like map:

p
(
Team1 wins
)
=
1
1
+
exp
⁡
(
a
Δ
r
+
b
)
,
p(Team1 wins)= 
1+exp(aΔr+b)
1
 ,
where you fit parameters 
a
,
b
a,b on historical games where both teams’ ranks are known.



In [295]:
X[X['Season'] < 2025].merge(
    y, 
    left_on=['Season', 'DayNum', 'TeamID'], 
    right_on=['Season', 'DayNum', 'TeamID'],
    how='left'
)[['ID', 'Season', 'TeamID', 'DayNum'] + SYS_BOOKIES]

,ID,Season,TeamID,DayNum,_pred_label,MOR,POM,DOK,SAG,MAS,WLK,WIL,DOL
0,NaN,2003,1102,37,NaN,110.0,104.0,NaN,114.0,90.0,103.0,NaN,NaN
1,NaN,2003,1102,44,NaN,106.0,84.0,NaN,78.0,56.0,52.0,NaN,NaN
2,NaN,2003,1102,51,NaN,127.0,126.0,NaN,113.0,100.0,52.0,NaN,NaN
3,NaN,2003,1102,57,NaN,124.0,156.0,NaN,127.0,137.0,121.0,NaN,NaN
4,NaN,2003,1102,65,NaN,119.0,134.0,NaN,124.0,120.0,116.0,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
181700,NaN,2024,1478,107,NaN,290.0,297.0,316.0,NaN,323.0,294.0,318.0,301.0
181701,NaN,2024,1478,114,NaN,316.0,313.0,319.0,NaN,335.0,316.0,333.0,319.0
181702,NaN,2024,1478,121,NaN,283.0,306.0,315.0,NaN,324.0,299.0,323.0,311.0
181703,NaN,2024,1478,128,NaN,281.0,300.0,307.0,NaN,322.0,302.0,322.0,311.0


In [293]:
y.groupby('Season').get_group(2025).groupby('TeamID')

# Massey

In [35]:
(
    male.massey_ordinals
    .assign(
        ID=lambda x: x['Season'].astype(str) + "_" + x['TeamID'].astype(str)
    )
    .groupby('ID')['OrdinalRank']
    .agg(
        OrdinalRankMu='mean',
        OrdinalRankStd='std',
        OrdinalRankMax='max',
        OrdinalRankMin='min'
    )
    .assign(
        OrdinalRankProb=lambda x: (x['OrdinalRankMax'] - x['OrdinalRankMin'])/(x['OrdinalRankMax'] + x['OrdinalRankMin'])
    )
    .reset_index()
)

,ID,OrdinalRankMu,OrdinalRankStd,OrdinalRankMax,OrdinalRankMin,OrdinalRankProb
0,2003_1102,144.287500,25.576263,219,52,0.616236
1,2003_1103,182.205000,25.591704,284,99,0.483029
2,2003_1104,27.655502,13.369710,106,1,0.981308
3,2003_1105,305.377500,13.428171,323,185,0.271654
4,2003_1106,243.265000,25.031104,307,166,0.298097
...,...,...,...,...,...,...
8351,2026_1477,286.547794,34.300729,362,125,0.486653
8352,2026_1478,299.031863,22.741362,362,206,0.274648
8353,2026_1479,297.580882,23.855615,361,93,0.590308
8354,2026_1480,285.339461,41.824069,362,106,0.547009


In [358]:
massey = male.massey_ordinals[
    (male.massey_ordinals['Season'] == 2003) ].copy()

In [359]:
massey_mat = (
    massey
    .rename(columns={'RankingDayNum': 'DayNum'})
    .merge(
        y[y['Season'] == 2003],
        right_on=['Season', 'DayNum', 'WTeamID'],
        left_on=['Season', 'DayNum', 'TeamID'],
        how='left'
    )
    .sort_values(by=['SystemName', 'OrdinalRank'])
    .pivot_table(
        index=['Season', 'TeamID', 'DayNum'],
        columns='SystemName',
        values='OrdinalRank'
    )
).fillna(999).values[:,4:]

In [360]:
massey_mat

array([[999., 999., 999., ..., 999., 999., 999.],
       [999., 999., 999., ..., 141., 999., 999.],
       [999., 999., 999., ..., 999., 999., 999.],
       ...,
       [291., 276., 266., ..., 272., 270., 999.],
       [999., 999., 999., ..., 999., 999., 999.],
       [291., 278., 264., ..., 272., 269., 271.]])

In [34]:
games.drop(columns=['ID']).groupby("DayNum").transform(lambda x: (x - x.mean()) / x.std())
# .corr()['Won'].sort_values(ascending=False)

,Won,Poss,OppPoss,OffRtg,DefRtg,PtsDiff,NetRtg,eFGPct,TOVPct,ORPct,FTRate
0,0.999122,0.837199,0.179911,-0.580784,-0.755659,0.328264,0.115113,-0.009337,1.914121,0.900592,-0.285824
1,0.999122,-0.179737,-0.266549,0.087629,-0.456881,0.382975,0.358428,-0.091934,-0.092595,0.765094,-0.309696
125978,-0.999122,0.179911,0.837199,-0.755659,-0.580784,-0.328264,-0.115113,-0.640961,1.015321,-0.023985,0.356470
125979,-0.999122,-0.266549,-0.179737,-0.456881,0.087629,-0.382975,-0.358428,-0.981730,-0.318965,1.171588,-0.358407
2,0.999834,-1.033933,-0.979186,0.767583,-0.315113,0.586478,0.665419,-0.058799,-0.718884,1.322762,0.683968
...,...,...,...,...,...,...,...,...,...,...,...
250502,-0.997805,0.932987,1.355393,0.987795,1.075986,-0.362235,-0.070112,0.241894,-0.675821,0.933453,-0.724597
250503,-0.997805,-0.360632,-0.519034,0.699961,1.701840,-0.724469,-0.796501,0.531296,-0.971162,0.302905,-1.806276
250504,-0.997805,0.325778,0.378579,-1.513475,-0.023096,-1.358380,-1.184863,-1.301581,0.840786,0.042198,0.149336
250505,-0.997805,-0.222030,-0.076827,-0.472628,0.280498,-0.724469,-0.598740,0.241894,0.168506,-0.826826,-0.120834


In [466]:
games.groupby("DayNum")['FTRate'].mean().sort_values(ascending=False)

DayNum
11     0.384141
105    0.380271
5      0.379665
4      0.379345
75     0.379111
         ...   
137    0.330392
144    0.327245
152    0.324307
140    0.258703
148    0.227336
Name: FTRate, Length: 148, dtype: float64

In [363]:
(
    massey
    .rename(columns={'RankingDayNum': 'DayNum'})
    .sort_values(by=['SystemName', 'OrdinalRank'])
    .pivot_table(
        index=['Season', 'TeamID', 'DayNum'],
        columns='SystemName',
        values='OrdinalRank'
    )
    .reset_index()
    .set_index(['Season', 'TeamID'])
    .pipe(
        lambda d: d.set_axis(
            d.index.map(lambda x: f"{x[0]}_{int(x[1])}"),
            axis=0
        ),
    )
    .rename_axis('ID')
)

SystemName,DayNum,AP,ARG,BIH,BOB,BRZ,COL,DOL,DUN,DWH,...,SAU,SE,SEL,STR,TSR,USA,WLK,WOB,WOL,WTE
ID,,,,,,,,,,,,,,,,,,,,,
2003_1102,35,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,159.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2003_1102,37,NaN,NaN,122.0,NaN,NaN,NaN,NaN,112.0,NaN,...,112.0,119.0,NaN,95.0,NaN,NaN,103.0,141.0,NaN,NaN
2003_1102,42,NaN,NaN,NaN,191.0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2003_1102,43,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,108.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2003_1102,44,NaN,NaN,122.0,NaN,NaN,NaN,NaN,NaN,NaN,...,60.0,96.0,NaN,103.0,NaN,NaN,52.0,102.0,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2003_1464,121,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2003_1464,126,NaN,278.0,NaN,276.0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,259.0,NaN,NaN,NaN,NaN,NaN,NaN,274.0
2003_1464,127,NaN,NaN,266.0,NaN,291.0,276.0,266.0,294.0,258.0,...,253.0,251.0,NaN,280.0,251.0,NaN,268.0,272.0,270.0,NaN
